# 추론 데모: YOLOv8s(위치 탐지, 1클래스 tile) + ResNet18 x2(숫자/색상 분류)

라벨 없이 이미지만 넣고 결과를 보는 노트북입니다 (`evaluate_models.ipynb`는 정답 라벨과 비교해서
점수를 내는 용도였고, 이건 그냥 "이 이미지에 뭐가 있는지 보여줘"용입니다).

## 파이프라인
1. YOLOv8s(`weights/yolov8s_tile_detector_best.pt`, `train_yolo_tile_only.ipynb`에서 학습한
   1클래스 tile 디텍터)로 이미지에서 타일 위치(박스)만 찾음
2. 각 박스를 margin 포함해서 crop
3. crop을 ResNet18 숫자 분류기 + 색상 분류기에 각각 통과시켜 최종 라벨 결정
   (YOLO 자체 클래스 예측은 1클래스(`tile`)뿐이라 참고용으로만 같이 표시하고, 최종 라벨은
   분류기 결과를 사용)
4. 박스+최종 라벨을 그린 이미지를 저장하고, 탐지 결과를 표로도 정리

## 대상 이미지
기본값은 `rummikub_data_3/images/test` 폴더의 이미지 전부입니다. 다른 폴더를 분석하고 싶으면
`TEST_IMAGE_DIR`만 바꾸면 됩니다.

## 출력
- `inference_results/annotated/{원본파일명}.jpg` — 박스+라벨이 그려진 이미지
- `inference_results/detections.csv` — 이미지별 박스 전체 목록 (좌표/신뢰도/최종 라벨)

In [ ]:
from __future__ import annotations

import csv
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

# ---- 경로 ----
WEIGHTS_DIR = Path("weights")
TEST_IMAGE_DIR = Path("rummikub_data_3/images/test")  # 분석하고 싶은 폴더로 바꿔도 됨
OUTPUT_DIR = Path("inference_results")
(OUTPUT_DIR / "annotated").mkdir(parents=True, exist_ok=True)

# ---- 파이프라인 설정 ----
YOLO_CONF = 0.25  # 이 신뢰도 미만 박스는 버림
YOLO_IMGSZ = 960
MARGIN_RATIO = 0.10  # classifier_number/classifier_color 만들 때와 동일하게 맞춤
IMG_SIZE = 96
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

image_paths = sorted(
    p for p in TEST_IMAGE_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
) if TEST_IMAGE_DIR.is_dir() else []
print(f"분석 대상: {len(image_paths)}장 ({TEST_IMAGE_DIR.resolve()})")
assert image_paths, f"이미지가 없습니다: {TEST_IMAGE_DIR.resolve()}"

## 모델 로드 (YOLOv8s tile 디텍터 + ResNet18 x2)

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO(str(WEIGHTS_DIR / "yolov8s_tile_detector_best.pt"))
yolo_class_names = [yolo_model.names[i] for i in sorted(yolo_model.names)]

eval_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def build_resnet_classifier(num_classes: int) -> nn.Module:
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)


def load_classifier(weights_path: Path):
    ckpt = torch.load(weights_path, map_location=DEVICE)
    classes = ckpt["classes"]
    model = build_resnet_classifier(len(classes))
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model, classes


number_model, number_classes = load_classifier(WEIGHTS_DIR / "resnet18_number_best.pt")
color_model, color_classes = load_classifier(WEIGHTS_DIR / "resnet18_color_best.pt")
print("number classes:", number_classes)
print("color classes:", color_classes)

## 파이프라인 함수: crop -> 분류 -> 최종 라벨 결합

In [ ]:
def crop_pixel_box_with_margin(image_rgb: np.ndarray, x1: float, y1: float, x2: float, y2: float):
    img_h, img_w = image_rgb.shape[:2]
    box_w, box_h = x2 - x1, y2 - y1
    margin_w, margin_h = box_w * MARGIN_RATIO, box_h * MARGIN_RATIO
    ex1 = int(max(0, x1 - margin_w))
    ey1 = int(max(0, y1 - margin_h))
    ex2 = int(min(img_w, x2 + margin_w))
    ey2 = int(min(img_h, y2 + margin_h))
    if ex2 <= ex1 or ey2 <= ey1:
        return None
    return image_rgb[ey1:ey2, ex1:ex2]


def classify_crop(model: nn.Module, classes: list[str], crop_rgb: np.ndarray) -> tuple[str, float]:
    tensor = eval_tf(Image.fromarray(crop_rgb)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
        pred_idx = int(probs.argmax())
    return classes[pred_idx], float(probs[pred_idx])


def combine_final_label(number_pred: str, color_pred: str) -> str:
    """둘 중 하나라도 joker로 판단하면 joker로 확정 (숫자 분류기가 조커의 무늬를
    가장 뚜렷하게 구분하므로 우선하되, 색상 분류기의 joker 판단도 존중)"""
    if number_pred == "joker" or color_pred == "joker":
        return "joker"
    return f"{color_pred}_{number_pred}"


def analyze_image(image_path: Path) -> tuple[np.ndarray, list[dict]]:
    result = yolo_model.predict(
        str(image_path),
        conf=YOLO_CONF,
        imgsz=YOLO_IMGSZ,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )[0]

    image_bgr = result.orig_img
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    annotated_bgr = image_bgr.copy()

    detections = []
    boxes_xyxy = result.boxes.xyxy.cpu().numpy()
    boxes_conf = result.boxes.conf.cpu().numpy()
    boxes_cls = result.boxes.cls.cpu().numpy().astype(int)

    for (x1, y1, x2, y2), yolo_conf, yolo_cls_id in zip(boxes_xyxy, boxes_conf, boxes_cls):
        crop = crop_pixel_box_with_margin(image_rgb, x1, y1, x2, y2)
        if crop is None or crop.size == 0:
            continue

        number_pred, number_conf = classify_crop(number_model, number_classes, crop)
        color_pred, color_conf = classify_crop(color_model, color_classes, crop)
        final_label = combine_final_label(number_pred, color_pred)

        detections.append(
            {
                "image": image_path.name,
                "x1": float(x1), "y1": float(y1), "x2": float(x2), "y2": float(y2),
                "yolo_class": yolo_class_names[yolo_cls_id] if yolo_cls_id < len(yolo_class_names) else "?",
                "yolo_conf": float(yolo_conf),
                "number_pred": number_pred, "number_conf": number_conf,
                "color_pred": color_pred, "color_conf": color_conf,
                "final_label": final_label,
            }
        )

        cv2.rectangle(annotated_bgr, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        label_text = f"{final_label} {min(number_conf, color_conf):.2f}"
        cv2.putText(
            annotated_bgr, label_text, (int(x1), max(0, int(y1) - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA,
        )

    return annotated_bgr, detections

## 전체 이미지 분석 실행

In [ ]:
all_detections: list[dict] = []

for image_path in image_paths:
    annotated_bgr, detections = analyze_image(image_path)
    out_path = OUTPUT_DIR / "annotated" / image_path.name
    cv2.imwrite(str(out_path), annotated_bgr)
    all_detections.extend(detections)
    print(f"{image_path.name}: 박스 {len(detections)}개 -> {out_path}")

csv_path = OUTPUT_DIR / "detections.csv"
fieldnames = [
    "image", "x1", "y1", "x2", "y2",
    "yolo_class", "yolo_conf",
    "number_pred", "number_conf",
    "color_pred", "color_conf",
    "final_label",
]
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_detections)

print(f"\n이미지 {len(image_paths)}장, 총 박스 {len(all_detections)}개")
print(f"주석 이미지: {(OUTPUT_DIR / 'annotated').resolve()}")
print(f"전체 목록 CSV: {csv_path.resolve()}")

## 미리보기 (선택)
노트북에서 바로 결과 이미지를 확인하고 싶으면 아래 셀을 실행하세요.

In [ ]:
import matplotlib.pyplot as plt

PREVIEW_COUNT = 3
for image_path in image_paths[:PREVIEW_COUNT]:
    annotated_bgr = cv2.imread(str(OUTPUT_DIR / "annotated" / image_path.name))
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB))
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

## 클래스별 탐지 개수 요약

In [ ]:
from collections import Counter

label_counts = Counter(d["final_label"] for d in all_detections)
for label, count in sorted(label_counts.items()):
    print(f"{label:<12} {count}")